In [1]:
import numpy as np
import sklearn

from tree import build_tree, most_common_label
from utils import load_preprocessed_ds

In [2]:
MIN_SAMPLES_SPLIT = 10
MAX_FEATURES = 3
N_TREES = 100
SEED = 42

In [3]:
X_tr, y_tr, X_te, y_te, X_tr_pd = load_preprocessed_ds()

In [4]:
rng = np.random.default_rng(44)

In [5]:
def bootstrap_sample(X, y, rng):
    n_samples = X.shape[0]
    sampled_indices = rng.choice(n_samples, size=n_samples, replace=True)

    X_boot = X[sampled_indices]
    y_boot = y[sampled_indices]

    bootstrap_indices = np.unique(sampled_indices)
    all_indices = np.arange(n_samples)
    sample_mask = np.isin(all_indices, bootstrap_indices)
    oob_indices = all_indices[~sample_mask]

    # print(len(oob_indices)/n_samples) # Should be ~ 0.37
    # print(np.intersect1d(bootstrap_indices, oob_indices)) # Should be empty
    return X_boot, y_boot, oob_indices

# bootstrap_sample(X_tr, y_tr, rng)

In [11]:
# Confirm overfitting when max_features = total features, and max_depth is unlimited
root = build_tree(X_tr, y_tr, rng=rng, max_features=20, depth=0, max_depth=None, min_samples_split=2)

train_preds = root.predict_batch(X_tr)
print(f"Train accuracy: {np.mean(train_preds == y_tr)}")

test_preds = root.predict_batch(X_te)
print(f"Test accuracy: {np.mean(test_preds == y_te)}")

Train accuracy: 1.0
Test accuracy: 0.7704918032786885


In [12]:
class RandomForest:
    def __init__(self, n_trees, max_features, max_depth=None, min_samples_split=MIN_SAMPLES_SPLIT, random_state=None):
        self.n_trees = n_trees
        self.max_features = max_features
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.rng = np.random.default_rng(random_state)
        self.trees = []
        self.oob_indices_per_tree = []

    def fit(self, X, y):
        for _ in range(self.n_trees):
            X_boot, y_boot, oob_idx = bootstrap_sample(X, y, self.rng)
            tree_root = build_tree(X_boot, y_boot, rng=self.rng, max_features=self.max_features, max_depth=self.max_depth, min_samples_split=self.min_samples_split)
            self.trees.append(tree_root)
            self.oob_indices_per_tree.append(oob_idx)

    def predict(self, X):
        all_preds = np.array([tree.predict_batch(X) for tree in self.trees])
        majority_vote = np.apply_along_axis(func1d=most_common_label, axis=0, arr=all_preds) # TODO: vectorize
        return majority_vote

In [13]:
def train_forest_get_acc(mode, n_trees, max_features, random_state):
    if mode == "custom":
        forest = RandomForest(n_trees=n_trees, max_features=max_features, random_state=random_state)
    elif mode == "sklearn":
        forest = sklearn.ensemble.RandomForestClassifier(n_estimators=n_trees, max_features=max_features,random_state=random_state)
    else:
        raise Exception("mode can be either 'custom' or 'sklearn'")

    forest.fit(X_tr, y_tr)
    preds = forest.predict(X_te)
    accuracy = np.mean(preds == y_te)
    return accuracy

In [16]:
# Compare with sklearn
custom_accs = list()
sklearn_accs = list()
for i in range(20):
    custom_accs.append(train_forest_get_acc("custom", n_trees=N_TREES, max_features=MAX_FEATURES, random_state=None))
    sklearn_accs.append(train_forest_get_acc("sklearn", n_trees=N_TREES, max_features=MAX_FEATURES, random_state=None))

custom_accs = np.array(custom_accs)
sklearn_accs = np.array(sklearn_accs)
print(f"custom: {custom_accs.mean():.4f} ± {custom_accs.std():.4f}")
print(f"sklearn: {sklearn_accs.mean():.4f} ± {sklearn_accs.std():.4f}")

custom: 0.8230 ± 0.0169
sklearn: 0.8311 ± 0.0201


In [15]:
# Confirm that 2 forests have the same trees if they use the same seed
forest1 = RandomForest(N_TREES, MAX_FEATURES, random_state=SEED)
forest1.fit(X_tr, y_tr)

forest2 = RandomForest(N_TREES, MAX_FEATURES, random_state=SEED)
forest2.fit(X_tr, y_tr)

assert forest1.trees[0].feature == forest2.trees[0].feature